In [13]:
%reset -f 

In [14]:
import numpy as np
import matplotlib.pyplot as plt
import torch

def region_tag(stau, zdis):
    s, tau = stau[:, 0], stau[:, 1]
    s_interface = torch.exp((-zdis*(zdis+1))/(2*(1/tau) + zdis - 1))
    return (s < s_interface).long()

In [15]:
n_s   = 400
n_tau = 200
d = torch.load(f'./data/2layer_interpol_ns{n_s}_aug.pt', weights_only=True)

stau   = d["stau"].double()
W      = d["W"].double()
kratio = d["kratio"].double()
zdis   = d["zdis"].double()

n_grid  = n_s * n_tau            # 45000 original grid rows
n_total = stau.shape[0]          # 45300
n_b    = 150    

# 1) geometric tag for everyone
tag = region_tag(stau, zdis).double()

# appended block = two stacked copies of the 150-pt interface
tag[n_grid : n_grid + n_b] = 0.0     # first copy  → 0
tag[n_grid + n_b :]        = 1.0     # second copy → 1


In [16]:
print(tag[n_grid:])

tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 

In [17]:
import os
os.makedirs('./data', exist_ok=True)

print("\n[6] SAVING TRAINING DATA (5 keys)")
# Create dictionary with 4 keys (all on CUDA)
training_dict = {
    'stau': stau,
    'W': W,
    'kratio': kratio,
    'zdis': zdis,
    'tag': tag
}

# Save to .pt file
dat_file = f'./data/2layer_interpol_ns{n_s}_tagged.pt'
torch.save(training_dict, dat_file)


[6] SAVING TRAINING DATA (5 keys)


In [18]:
stau.shape

torch.Size([80400, 2])